## Cell-type annotation of Xenium dataset

- Manual annotation of clusters & sub-clusterse
- Automated: astir

In [ ]:
import os
import sys
import gc
import cv2
import gzip

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq

import torch
import tifffile
import xml.etree.ElementTree as ET

from collections import OrderedDict
from skimage.transform import rescale
from IPython.display import display


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager
from matplotlib import rcParams

rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

In [ ]:
sys.path.append('../')
sys.path.append('../util')
import IO, utils

In [ ]:
# Helper function
def disp_cell_types(adata, ncol=3):
    assert 'cell_type' in adata.obs_keys(), "Please perform cell-type annotations first"
    
    cell_types = sorted(adata.obs.cell_type.unique())
    cell_types = np.setdiff1d(cell_types, 'Other')
    
    nrow = len(cell_types) // ncol 
    if len(cell_types)%ncol != 0:
        nrow += 1
    
    fig, axes = plt.subplots(nrow, ncol, figsize=(4*ncol, 2*nrow))
    idx = 0
    for r in range(nrow):
        for c in range(ncol):
            if idx >= len(cell_types):
                axes[r, c].axis('off')
                continue
                
            ax = axes[r, c]
            ax = sc.pl.umap(adata, color='cell_type', groups=[cell_types[idx]],ax=ax, frameon=False, show=False)
            idx += 1
    
    plt.tight_layout()
    plt.show()

    return None

### Load dataset

In [ ]:
data_path = '../data/xenium/'
sample_ids = [f for f in sorted(os.listdir(data_path))
              if os.path.isdir(os.path.join(data_path, f))]

In [ ]:
adata_list = []
for sample_id in sample_ids:
    adata = IO.load_xenium(os.path.join(data_path, sample_id))
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    sc.pp.pca(adata)
    sc.pp.neighbors(adata, use_rep='X_pca')
    sc.tl.umap(adata)

    adata_list.append(adata)

del sample_id, adata

In [ ]:
# Save filtered & normalized adata back to original directories
for adata, sample_id in zip(adata_list, sample_ids):
    adata.write_h5ad(os.path.join(data_path, sample_id, 'filtered_feature_matrix.h5'))

### Annotation

#### Manual annotation w/ Leiden:

#### Astir

Try both full-panel annotation & customized annotation

In [ ]:
import astir
import yaml
from astir.data import from_csv_yaml, from_anndata_yaml


Construct marker yaml:

In [ ]:
# Full-panel
annot_df = pd.read_csv('../data/Xenium_hMulti_v1_metadata.csv', index_col=[0,1,2,3])
annot_df.fillna(0, inplace=True)
annot_df.replace('x', 1, inplace=True)

In [ ]:
# Select non-overlapping cell types
cell_types = [
    'Hepatocyte', 'T cells', 'B  cells',
    'Dendritic cells', 'Monocyte', 'Endothelial', 
    'Smooth muscle', 'Fibroblast', 'Macrophages',
    'NK cells', 'HSC', 'Erythroid', 'Cholangiocyte', 
    'Granulocytes', 'Breast Adipocyte'
]

annot_df = annot_df[cell_types]
annot_df.astype(np.uint8)
annot_df.sum(0)

In [ ]:
cell_type_dict = {
    'Cell_types': {
        cell_type: annot_df.loc[annot_df[cell_type] == 1].index.get_level_values(0).tolist()
        for cell_type in cell_types
    }
}

# outdir = '../data/'
# filename = 'xenium_panel_markers.yml'

# with open(os.path.join(outdir, filename), 'w') as ofile:
#     yaml.dump(cell_type_dict, ofile, default_flow_style=False)


In [ ]:
# Customized
cell_type_dict = {
    'Cell_types': {
        'PP_Hep': ['CYP2A7', 'HPX', 'HMGCS2', 'CYP2B6'],
        'PC_Hep': ['CYP1A1', 'CYP3A4', 'APOA5', 'ADH4'],
        'Cholangiocyte': ['KRT7', 'CFTR', 'EPCAM'],
        'Fibroblast': ['FBN1', 'PDGFRA', 'PDGFRB', 'THY1'],
        'Smooth Muscle': ['MYH11', 'ACTA2', 'CNN1'],
        'Endothelial': ['PECAM1', 'CD34', 'SNCG'],
        'Sinusoidal': ['LYVE1', 'PDPN'],
        'M1 Macrophage': ['CD68', 'CD14'],
        'M2 Macrophage': ['CD163', 'MARCO'],
        'B-cells': ['CD19', 'PTPRC'],
        'CD4 T-cells': ['CD3D', 'CD3E', 'CD4'],
        'CD8 T-cells': ['CD3D', 'CD3E', 'CD8A'],
    }
}

cell_types = list(cell_type_dict['Cell_types'].keys())
markers = []
for _, v in cell_type_dict['Cell_types'].items():
    markers.extend(v)
del v

# outdir = '../data/'
# filename = 'xenium_custom_markers.yml'

# with open(os.path.join(outdir, filename), 'w') as ofile:
#     yaml.dump(cell_type_dict, ofile, default_flow_style=False)


Load expr & marker yaml:

In [ ]:
# transform to csv?
data_path = '../data/xenium/'
sample_ids = [f for f in sorted(os.listdir(data_path))
              if os.path.isdir(os.path.join(data_path, f))]
sample_id = sample_ids[9]
print('Loading sample {}...'.format(sample_id))
adata = IO.load_xenium(os.path.join(data_path, sample_id), raw_count=False)

# df = adata.to_df()
# df.to_csv('../data/xenium/{}/filtered_feature_matrix.csv'.format(sample_id), index=True)

In [ ]:
ast = from_csv_yaml(csv_input='../data/xenium/{}/filtered_feature_matrix.csv'.format(sample_id),
                    marker_yaml='../data/xenium_custom_markers.yml')
ast.fit_type(batch_size=1024, n_init=3, max_epochs=100)

In [ ]:
soft_assignments = ast.get_celltype_probabilities()
soft_assignments.max(0)

In [ ]:
assignments = ast.get_celltypes(threshold=0.25)
adata.obs['cell_type'] = assignments
adata.obs['cell_type'].value_counts()

In [ ]:
# Visualize Cell-type annotations
disp_cell_types(adata)

In [ ]:
print('Saving sample {}...'.format(sample_id))
adata.write_h5ad('../data/xenium/{}/filtered_feature_matrix.h5'.format(sample_id))

### Evaluation

In [ ]:
import anndata as ad

Comparison across samples

In [ ]:
cell_type_dict = {
    'Cell_types': {
        'PP_Hep': ['CYP2A7', 'HPX', 'HMGCS2', 'CYP2B6'],
        'PC_Hep': ['CYP1A1', 'CYP3A4', 'APOA5', 'ADH4'],
        'Cholangiocyte': ['KRT7', 'CFTR', 'EPCAM'],
        'Fibroblast': ['FBN1', 'PDGFRA', 'PDGFRB', 'THY1'],
        'Smooth Muscle': ['MYH11', 'ACTA2', 'CNN1'],
        'Endothelial': ['PECAM1', 'CD34', 'SNCG'],
        'Sinusoidal': ['LYVE1', 'PDPN'],
        'M1 Macrophage': ['CD68', 'CD14'],
        'M2 Macrophage': ['CD163', 'MARCO'],
        'B-cells': ['CD19', 'PTPRC'],
        'CD4 T-cells': ['CD3D', 'CD3E', 'CD4'],
        'CD8 T-cells': ['CD3D', 'CD3E', 'CD8A'],
    }
}

cell_types = list(cell_type_dict['Cell_types'].keys())
markers = []
for _, v in cell_type_dict['Cell_types'].items():
    markers.extend(v)
del v

In [ ]:
# Load cell type annotations
data_path = '../data/xenium/'
sample_ids = [f for f in sorted(os.listdir(data_path))
              if os.path.isdir(os.path.join(data_path, f))]

adatas = []
for sample_id in sample_ids:
    adata = IO.load_xenium(os.path.join(data_path, sample_id), raw_count=False)
    adatas.append(adata)

adata_cat = ad.concat(adatas)
adata_cat.shape

gc.collect()

In [ ]:
# Marker visualization
from scipy.stats import zscore

adata_markers = adata_cat[
    adata_cat.obs['cell_type'].isin(cell_types),
    markers
]

marker_expr = np.zeros((len(cell_types), len(markers)))
for i, cell_type in enumerate(cell_types):
    marker_expr[i] = adata_markers[adata_markers.obs['cell_type'] == cell_type].X.A.mean(0)


In [ ]:
# Per-sample
for sample_id, adata in zip(sample_ids, adatas):
    adata_markers = adata[adata.obs['cell_type'].isin(cell_types), markers]
    marker_expr = np.zeros((len(cell_types), len(markers)))

    for i, cell_type in enumerate(cell_types):
        marker_expr[i] = adata_markers[adata_markers.obs['cell_type'] == cell_type].X.A.mean(0)

    marker_expr = zscore(marker_expr.T, axis=1)
    marker_expr_df = pd.DataFrame(
        marker_expr,
        index=markers,
        columns=cell_types
    )

    fig, ax = plt.subplots(figsize=(5, 7))
    g = sns.heatmap(marker_expr_df, cmap='seismic', ax=ax)
    g.set_xticklabels(g.get_xticklabels(), rotation=75)
    plt.title(sample_id)
    plt.show()
  

del sample_id, adata

In [ ]:
marker_expr = zscore(marker_expr.T, axis=1)  # dim: [# markers, # cell types]
marker_expr_df = pd.DataFrame(
    marker_expr, 
    index=markers,
    columns=cell_types
)

fig, ax = plt.subplots(figsize=(5, 7))
g = sns.heatmap(marker_expr_df, cmap='seismic', ax=ax)
g.set_xticklabels(g.get_xticklabels(), rotation=75)
plt.title('All samples')
plt.show()

marker_expr_df.to_csv('../results/marker_gexp_zscore.csv', index=True)

del adata_markers, marker_expr, marker_expr_df

In [ ]:
marker_expr_df.to_csv('../results/marker_gexp_zscore.csv', index=True)

In [ ]:
# Individual adata?
for i, adata in enumerate(adatas):
    adata_markers = adata[
        adata.obs['cell_type'].isin(cell_types),
        markers
    ]
    
    marker_expr = np.zeros((len(cell_types), len(markers)))
    for j, cell_type in enumerate(cell_types):
        marker_expr[j] = adata_markers[adata_markers.obs['cell_type'] == cell_type].X.A.mean(0)

    marker_expr_df = pd.DataFrame(marker_expr, 
                                  index=cell_types,
                                  columns=markers)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    g = sns.heatmap(marker_expr_df, cmap='seismic', ax=ax)
    g.set_xticklabels(g.get_xticklabels(), rotation=75)
    plt.title(sample_ids[i])
    plt.show()

del adata, adata_markers, marker_expr, marker_expr_df, cell_type

In [ ]:
gc.collect()

In [ ]:
# Visualize annotations
for sample_id, adata in zip(sample_ids:
    print('=== {} ==='.format(sample_id))
    sc.pl.umap(adata, color='cell_type')
    disp_cell_types(adata, ncol=3)
    print('\n\n')
    
del sample_id, adata

In [ ]:
all_annotations = []

for sample_id in sample_ids:
    annot_df = IO.load_xenium(os.path.join(data_path, sample_id), raw_count=False).obs[['cell_type']]
    annot_df['Sample_ID'] = sample_id
    annot_df['Gender'] = 'M' if 'M' in sample_id else 'F'
    all_annotations.append(annot_df)

del sample_id

In [ ]:
all_annotations_df = pd.concat(all_annotations)
all_annotations_df['Gender'] = all_annotations_df['Sample_ID'].apply(lambda x: 'M' if 'M' in x else 'F')

In [ ]:
ax = (all_annotations_df.groupby('Sample_ID')['cell_type'].value_counts(normalize=True)
 .unstack('cell_type').iloc[:,:-1]
 .plot.bar(stacked=True, figsize=(6, 4), cmap='tab20')
)

ax.spines[['right', 'top']].set_visible(False)
ax.get_xaxis().tick_bottom()
ax.get_yaxis().tick_left()

plt.legend(loc='center right', bbox_to_anchor=(1.5, 0.5))
plt.show()


In [ ]:
ax = (all_annotations_df.groupby('Gender')['cell_type'].value_counts(normalize=True)
 .unstack('cell_type').iloc[:,:-1]
 .plot.bar(stacked=True, figsize=(6, 4), cmap='tab20')
)

ax.spines[['right', 'top']].set_visible(False)
ax.get_xaxis().tick_bottom()
ax.get_yaxis().tick_left()

plt.legend(loc='center right', bbox_to_anchor=(1.5, 0.5))
plt.show()

---